# 第14章 无监督学习 —— 配套代码

对应正文 `book/part3/14-unsupervised.md`。无标签学习：降维（PCA）、聚类（KMeans/层次）、异常检测、市场状态识别。

> 运行前请先生成内置数据：`uv run python scripts/make_sample_data.py`。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装。")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fds import load_sample_prices, load_market, daily_returns, set_chinese_font

set_chinese_font()
rets = daily_returns(load_sample_prices())
print('日收益形状:', rets.shape)

## 1. 降维：主成分分析（PCA）

对四只股票的日收益做 PCA，第一主成分通常解释最大方差，对应「市场因子」（各股同涨同跌）。

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(rets.to_numpy())
ev = pca.explained_variance_ratio_
print('各主成分解释方差占比:', np.round(ev, 3))
print('第一主成分载荷（同号→市场因子）:', dict(zip(rets.columns, np.round(pca.components_[0], 3))))

## 2. 聚类：KMeans 把股票分组

内置只有 4 只股票，聚类不充分。这里**合成 30 只股票**（3 个真实分组）演示聚类能否还原分组。

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(0)
T = 500
factors = rng.normal(0, 0.01, (T, 3))          # 3 个分组因子
true_group = np.repeat([0, 1, 2], 10)            # 30 只股票真实分组
R = np.column_stack([factors[:, g] + rng.normal(0, 0.008, T) for g in true_group])
df = pd.DataFrame(R, columns=[f'S{i:02d}' for i in range(30)])
# 特征：年化收益、年化波动、与等权市场的相关
mkt = df.mean(axis=1)
feat = pd.DataFrame({'ret': df.mean() * 252, 'vol': df.std() * np.sqrt(252),
                     'corr': df.corrwith(mkt)})
labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(
    StandardScaler().fit_transform(feat))
print('调整兰德指数（1=完全还原分组）:', round(adjusted_rand_score(true_group, labels), 3))

In [ ]:
# 在前两个主成分上可视化聚类结果
xy = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(feat))
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap='viridis', s=60)
ax.set_title('KMeans 聚类结果（特征经 PCA 投影到 2 维）')
ax.set_xlabel('主成分1'); ax.set_ylabel('主成分2')
plt.tight_layout(); plt.show()

## 3. 层次聚类：按相关性给资产分群

用 1−相关系数作距离，对四只内置股票做层次聚类，看哪些资产「抱团」。

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

dist = 1 - rets.corr()
Z = linkage(squareform(dist.to_numpy(), checks=False), method='average')
fig, ax = plt.subplots(figsize=(8, 4))
dendrogram(Z, labels=list(rets.columns), ax=ax)
ax.set_title('四只股票的层次聚类（按 1−相关系数）')
ax.set_ylabel('距离')
plt.tight_layout(); plt.show()

## 4. 异常检测：识别极端交易日

用孤立森林（Isolation Forest）在 TECH 日收益上标记离群点——可用于异常交易/风控预警。

In [ ]:
from sklearn.ensemble import IsolationForest

x = rets['TECH'].to_numpy().reshape(-1, 1)
flag = IsolationForest(contamination=0.03, random_state=0).fit_predict(x)
outliers = rets['TECH'][flag == -1]
print(f'标记出 {len(outliers)} 个异常日；最极端几日:')
print(outliers.reindex(outliers.abs().sort_values(ascending=False).index).head().round(4))

## 5. 市场状态识别（regime detection）

把每个交易日用「当日收益 + 滚动波动率」表示，KMeans 聚成「平静 / 动荡」两态。

In [ ]:
r = load_market()['index_return'].dropna()
feat2 = pd.DataFrame({'r': r, 'vol': r.rolling(20).std()}).dropna()
lab = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(
    StandardScaler().fit_transform(feat2))
feat2['regime'] = lab
summary = feat2.groupby('regime').agg(平均波动=('vol', 'mean'), 天数=('r', 'size'))
print(summary.round(4))
print('波动更高的那一类即「动荡期」。')

## 习题参考解答

**习题1：用 PCA 把四只股票降到 2 维并散点可视化。**

In [ ]:
xy4 = PCA(n_components=2).fit_transform(rets.T.to_numpy())
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(xy4[:, 0], xy4[:, 1], s=80, color='#2E5A88')
for i, name in enumerate(rets.columns):
    ax.annotate(name, (xy4[i, 0], xy4[i, 1]), textcoords='offset points', xytext=(6, 6))
ax.set_title('四只股票在主成分空间的位置'); plt.tight_layout(); plt.show()

**习题2：把合成股票池的 KMeans 簇数从 3 改为 2 和 4，比较调整兰德指数。**

In [ ]:
Xs = StandardScaler().fit_transform(feat)
for k in (2, 3, 4):
    lab_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xs)
    print(f'k={k}: 调整兰德指数 = {adjusted_rand_score(true_group, lab_k):.3f}')